# CombBandits — All Experiments

**Runtime:** GPU → H100 (preferred) or A100 + enable **High-RAM**

**Secrets to set** (key icon in left sidebar):
| Secret | Value |
|---|---|
| `HF_TOKEN` | HuggingFace token (for Llama 4 Scout) |
| `AWS_ACCESS_KEY_ID` | AWS key (for Bedrock + S3) |
| `AWS_SECRET_ACCESS_KEY` | AWS secret |

Then run all cells top-to-bottom. Crash recovery is automatic — re-run from any cell.

In [ ]:
# ── 0. GPU check ──────────────────────────────────────────────────────────────
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), 'Switch to GPU runtime first (Runtime > Change runtime type)'
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'CUDA: {torch.cuda.get_device_name(0)}  |  VRAM: {vram:.0f} GB')

In [ ]:
# ── 1. Credentials ────────────────────────────────────────────────────────────
# Set these three Colab Secrets (key icon, left sidebar):
#   AWS_ACCESS_KEY_ID     — see project README / setup notes
#   AWS_SECRET_ACCESS_KEY — see project README / setup notes
#   HF_TOKEN              — https://huggingface.co/settings/tokens
import os
from google.colab import userdata

AWS_ACCESS_KEY_ID     = userdata.get('AWS_ACCESS_KEY_ID')     or ''
AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_ACCESS_KEY') or ''
HF_TOKEN              = userdata.get('HF_TOKEN')              or ''
AWS_REGION            = 'us-east-1'
S3_BUCKET             = 'combbandits-results-099841456154'

os.environ['AWS_ACCESS_KEY_ID']      = AWS_ACCESS_KEY_ID
os.environ['AWS_SECRET_ACCESS_KEY']  = AWS_SECRET_ACCESS_KEY
os.environ['AWS_DEFAULT_REGION']     = AWS_REGION
os.environ['HF_TOKEN']               = HF_TOKEN

print('HF_TOKEN:',  'SET' if HF_TOKEN  else 'MISSING')
print('AWS creds:', 'SET' if AWS_ACCESS_KEY_ID else 'MISSING')

# Quick Bedrock sanity check — catches auth issues immediately
import boto3
resp = boto3.client('bedrock-runtime', region_name='us-east-1').converse(
    modelId='us.meta.llama4-scout-17b-instruct-v1:0',
    messages=[{'role':'user','content':[{'text':'Say OK.'}]}],
    inferenceConfig={'maxTokens': 5, 'temperature': 0.0},
)
print('Bedrock:', resp['output']['message']['content'][0]['text'].strip())

In [ ]:
# ── 2. Clone repo + install ───────────────────────────────────────────────────
import sys
from pathlib import Path

REPO = Path('/content/CombBandits')
if not REPO.exists():
    !git clone https://github.com/vkmk1/CombBandits {REPO}
else:
    !git -C {REPO} pull --ff-only

%cd {REPO}
if str(REPO/'src') not in sys.path:
    sys.path.insert(0, str(REPO/'src'))

!pip install -q -e '.[dev]'
!pip install -q 'transformers>=4.47' accelerate bitsandbytes huggingface_hub boto3 scikit-learn
print('Done.')

In [ ]:
# ── 3. CPU experiments in parallel (exp4, exp5, exp6, exp7) ──────────────────
# exp9_bedrock runs here too if AWS creds are set.
# Each experiment manages its own ProcessPoolExecutor internally.
# Crash recovery: checkpoint JSON written every 10 tasks; re-run resumes.
import time, threading, os
from combbandits.engine.runner import ExperimentRunner

cpu_cores = os.cpu_count() or 8
EXPS = [
    ('exp4_mind',           max(1, cpu_cores // 8)),
    ('exp5_influence_max',  max(1, cpu_cores // 8)),
    ('exp7_ablation_trust', max(1, cpu_cores // 4)),
    ('exp6_workshop_main',  max(4, cpu_cores // 2)),
]
if AWS_ACCESS_KEY_ID:
    EXPS.append(('exp9_bedrock', 4))

errors = {}

def _run(name, workers):
    try:
        Path(f'results/{name}').mkdir(parents=True, exist_ok=True)
        t0 = time.time()
        ExperimentRunner(f'configs/experiments/{name}.yaml', f'results/{name}').run(max_workers=workers)
        print(f'  [{name}] done in {time.time()-t0:.0f}s')
    except Exception as e:
        errors[name] = str(e)
        print(f'  [{name}] FAILED: {e}')

threads = []
for name, w in EXPS:
    t = threading.Thread(target=_run, args=(name, w), daemon=True)
    t.start(); threads.append((name, t)); time.sleep(1)

for name, t in threads:
    t.join()
    print(name, '→', 'OK' if name not in errors else f'FAILED: {errors[name]}')

In [ ]:
# ── 4. exp8 — GPU batched simulation (all seeds in parallel on CUDA) ──────────
import yaml, json, time, torch
from combbandits.gpu.batched_trial import run_batched_experiment

with open('configs/experiments/exp8_scaling_d.yaml') as f:
    cfg = yaml.safe_load(f)

device = torch.device('cuda')
print(f'Running exp8_scaling_d on {device}...')
t0 = time.time()
results_8 = run_batched_experiment(cfg, device=device)
print(f'Done in {(time.time()-t0)/60:.1f} min  ({len(results_8)} trials)')

Path('results/exp8_scaling_d').mkdir(parents=True, exist_ok=True)
with open('results/exp8_scaling_d/exp8_scaling_d_results.json', 'w') as f:
    json.dump(results_8, f, indent=2)

In [ ]:
# ── 5. exp9_local — Llama 4 Scout + weight tracking ──────────────────────────
# Skipped if no HF_TOKEN or VRAM < 20 GB.
import yaml, torch
from combbandits.engine.runner import ExperimentRunner

vram = torch.cuda.get_device_properties(0).total_memory / 1e9
if not HF_TOKEN:
    print('Skipping exp9_local: no HF_TOKEN.')
elif vram < 20:
    print(f'Skipping exp9_local: only {vram:.0f} GB VRAM (need ≥20 GB).')
else:
    # Inject HF token into config
    cfg_path = 'configs/experiments/exp9_local.yaml'
    with open(cfg_path) as f: cfg = yaml.safe_load(f)
    cfg['oracles'][0]['hf_token'] = HF_TOKEN
    with open(cfg_path, 'w') as f: yaml.dump(cfg, f)

    Path('results/exp9_local').mkdir(parents=True, exist_ok=True)
    Path('metadata').mkdir(exist_ok=True)
    t0 = time.time()
    # workers=1: model stays loaded between trials
    ExperimentRunner(cfg_path, 'results/exp9_local').run(max_workers=1)
    print(f'exp9_local done in {(time.time()-t0)/60:.1f} min')

In [ ]:
# ── 6. Figures for all experiments ───────────────────────────────────────────
from combbandits.analysis.plots import generate_all_figures

for exp in ['exp4_mind','exp5_influence_max','exp6_workshop_main',
            'exp7_ablation_trust','exp8_scaling_d','exp9_bedrock','exp9_local']:
    rp = Path(f'results/{exp}/{exp}_results.json')
    if rp.exists():
        try:
            generate_all_figures(str(rp), f'figures/{exp}')
            print(f'  {exp}: figures saved')
        except Exception as e:
            print(f'  {exp}: plot error: {e}')

In [ ]:
# ── 7. Weight tracking analysis (exp9_local only) ────────────────────────────
import sqlite3, json, numpy as np, matplotlib.pyplot as plt, pandas as pd
from sklearn.decomposition import PCA

db_path = Path('metadata/oracle_weights.db')
if db_path.exists():
    db    = sqlite3.connect(str(db_path))
    calls  = pd.read_sql('SELECT * FROM oracle_calls  ORDER BY call_id',  db)
    groups = pd.read_sql('SELECT * FROM query_groups  ORDER BY group_id', db)
    db.close()
    print(f'{len(calls)} oracle calls, {len(groups)} query groups')

    primary = calls[calls['query_variant'] == 0].copy()
    Path('figures/exp9_local').mkdir(parents=True, exist_ok=True)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes[0,0].plot(primary['trial_round'], primary['output_entropy'],     alpha=.7, color='steelblue')
    axes[0,0].set(title='Output entropy', xlabel='Round')
    axes[0,1].plot(primary['trial_round'], primary['attn_on_metadata'],   alpha=.7, color='crimson')
    axes[0,1].set(title='Attention on arm metadata', xlabel='Round')
    axes[1,0].plot(primary['trial_round'], primary['suggestion_logprob'], alpha=.7, color='green')
    axes[1,0].set(title='Log-prob of suggestion', xlabel='Round')
    axes[1,1].plot(groups['trial_round'], groups['kappa'],         label='κ', color='orange')
    axes[1,1].plot(groups['trial_round'], groups['hidden_kl_div'], label='hidden div', color='purple')
    axes[1,1].set(title='Output κ vs internal diversity', xlabel='Round'); axes[1,1].legend()
    plt.suptitle('Llama 4 Scout Oracle — Weight Tracking', fontsize=13)
    plt.tight_layout()
    plt.savefig('figures/exp9_local/weight_tracking.png', dpi=150); plt.show()

    hidden_vecs = np.array([json.loads(r) for r in primary['hidden_state_pca']])
    if len(hidden_vecs) >= 3:
        pca = PCA(n_components=2)
        coords = pca.fit_transform(hidden_vecs)
        fig, ax = plt.subplots(figsize=(8, 6))
        sc = ax.scatter(coords[:,0], coords[:,1], c=primary['trial_round'].values,
                        cmap='viridis', alpha=.7, s=20)
        plt.colorbar(sc, label='Bandit round')
        ax.set(title='Hidden state PCA — representation drift over learning',
               xlabel=f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)',
               ylabel=f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
        plt.tight_layout()
        plt.savefig('figures/exp9_local/hidden_state_pca.png', dpi=150); plt.show()
else:
    print('No weight tracking DB (exp9_local was not run).')

In [ ]:
# ── 8. Upload to S3 ───────────────────────────────────────────────────────────
if AWS_ACCESS_KEY_ID:
    import boto3
    s3 = boto3.client('s3')
    uploaded = 0
    for folder in ['results', 'figures', 'metadata']:
        for root, _, files in os.walk(folder):
            for fname in files:
                full = os.path.join(root, fname)
                s3.upload_file(full, S3_BUCKET, full)
                uploaded += 1
    print(f'Uploaded {uploaded} files to s3://{S3_BUCKET}/')
else:
    print('No AWS creds — download from Files panel: /content/CombBandits/{results,figures,metadata}/')